# WRDS + OptionMetrics Exploration

Professor-facing notebook for the strongest validated story: option-implied features improve supervised prediction of future stock-level volatility stress, and the resulting probabilities are useful for cross-sectional risk ranking.

## 1. Problem

- **Task:** predict whether a stock will enter a high-volatility regime over the next 20 trading days.
- **Main comparison:** stock-only vs option-only vs stock + option + surface.
- **Main usefulness test:** do higher predicted-risk buckets actually realize higher future volatility?

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd()
if ROOT.name != 'wrds_optionmetrics_exploration':
    ROOT = ROOT / 'wrds_optionmetrics_exploration'

METRICS = ROOT / 'outputs' / 'metrics'
SUMMARIES = ROOT / 'outputs' / 'summaries'

surface_metrics = pd.read_csv(METRICS / 'surface_extension_metrics.csv')
bucket_metrics = pd.read_csv(METRICS / 'phase2_bucket_analysis_metrics.csv')
bucket_diag = pd.read_csv(METRICS / 'phase2_bucket_analysis_diagnostics.csv', parse_dates=['trade_date'])

extract_stock = json.loads((SUMMARIES / 'extract_stock_data_summary.json').read_text())
build_stock = json.loads((SUMMARIES / 'build_stock_panel_summary.json').read_text())
extract_option = json.loads((SUMMARIES / 'extract_option_data_summary.json').read_text())
build_option = json.loads((SUMMARIES / 'build_option_features_summary.json').read_text())
build_surface = json.loads((SUMMARIES / 'build_surface_factors_summary.json').read_text())


## 2. Data and target quality

In [ ]:
quality_table = pd.DataFrame([
    {'Item': 'Universe size', 'Value': extract_stock['universe_size']},
    {'Item': 'Resolved tickers', 'Value': extract_stock['resolved_permnos']},
    {'Item': 'Stock rows', 'Value': extract_stock['stock_daily_rows']},
    {'Item': 'Merged panel rows', 'Value': build_option['merged_panel_rows']},
    {'Item': 'Complete-case rows', 'Value': build_option['complete_case_rows']},
    {'Item': 'Surface-extension rows', 'Value': build_surface['surface_extension_rows']},
    {'Item': 'Train threshold for future RV(20d)', 'Value': round(build_stock['train_threshold_future_rv_20d'], 4)},
])
quality_table


## 3. Main predictive comparison

In [ ]:
keep_models = [
    'stock_only_logreg_surface_common',
    'option_only_logreg_surface_common',
    'all_features_logreg',
]
main_table = surface_metrics[(surface_metrics['split'] == 'test') & (surface_metrics['model'].isin(keep_models))].copy()
main_table['label'] = main_table['model'].map({
    'stock_only_logreg_surface_common': 'Stock Only',
    'option_only_logreg_surface_common': 'Option Only',
    'all_features_logreg': 'Stock + Option + Surface',
})
main_table[['label', 'macro_f1', 'balanced_accuracy', 'auroc', 'pr_auc']].sort_values('pr_auc', ascending=False)


In [ ]:
fig = px.bar(
    main_table.sort_values('pr_auc'),
    x='pr_auc',
    y='label',
    orientation='h',
    text='pr_auc',
    title='Test PR-AUC: original fixed-split logistic comparison',
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.show()


## 4. Bucket usefulness

In [ ]:
label_map = {
    'stock_only_prob': 'Stock Only',
    'option_only_prob': 'Option Only',
    'all_features_prob': 'Stock + Option + Surface',
}
core_bucket = bucket_metrics[(bucket_metrics['split'] == 'test') & (bucket_metrics['signal_column'].isin(label_map))].copy()
core_bucket['signal_name'] = core_bucket['signal_column'].map(label_map)

core_diag = bucket_diag[(bucket_diag['split'] == 'test') & (bucket_diag['signal_column'].isin(label_map))].copy()
core_diag['signal_name'] = core_diag['signal_column'].map(label_map)

bucket_summary = core_diag.groupby(['signal_name'], as_index=False).agg(
    avg_rank_ic_future_rv=('daily_rank_ic_future_rv', 'mean'),
    avg_top_bottom_hit_rate_spread=('top_bottom_hit_rate_spread', 'mean'),
)
bucket_summary.sort_values('avg_rank_ic_future_rv', ascending=False)


In [ ]:
fig = px.line(
    core_bucket,
    x='bucket',
    y='average_future_rv_20d',
    color='signal_name',
    markers=True,
    title='Future 20-day realized volatility by predicted-risk bucket',
)
fig.update_yaxes(tickformat='.0%')
fig.show()


## 5. Benchmark robustness

In [ ]:
benchmark_models = [
    'all_features_logreg',
    'all_features_xgb_surface_common',
    'all_features_trail2y_logreg_surface_common',
    'all_features_trail5y_logreg_surface_common',
]
bench = surface_metrics[(surface_metrics['split'] == 'test') & (surface_metrics['model'].isin(benchmark_models))].copy()
bench['label'] = bench['model'].map({
    'all_features_logreg': 'Baseline logistic',
    'all_features_xgb_surface_common': 'XGBoost',
    'all_features_trail2y_logreg_surface_common': '2Y trailing logistic',
    'all_features_trail5y_logreg_surface_common': '5Y trailing logistic',
})
bench[['label', 'macro_f1', 'pr_auc']].sort_values('pr_auc', ascending=False)


## 6. Limitations

- The headline model is deliberately simple and interpretable.
- XGBoost and trailing-window retrains are useful benchmarks, but they do not clearly beat the original option-driven result on the Phase 2 ranking objective.
- The portfolio overlay and text/news branches were weaker than the core risk-ranking result and are therefore not the main presentation path.